<a href="https://colab.research.google.com/github/Ryuta-Y/cv-practice/blob/main/optical_flow_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import importlib.util
import json
import os
import random
import shlex
import subprocess
import sys
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch


def in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False


def ensure_packages(packages: dict[str, str]) -> None:
    missing_specs = [spec for mod, spec in packages.items() if importlib.util.find_spec(mod) is None]
    if not missing_specs:
        return

    pip_args = []
    for spec in missing_specs:
        pip_args.extend(shlex.split(spec))

    print("不足パッケージをインストールします:")
    for pkg in pip_args:
        print(" -", pkg)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pip_args])


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def print_runtime_info() -> None:
    device = get_device()
    print(f"実行デバイス: {device}")
    if device.type == "cuda":
        print("GPU 名:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)


def get_output_dir(name: str, prefer_drive: bool = False) -> Path:
    if prefer_drive and in_colab() and Path("/content/drive/MyDrive").exists():
        root = Path("/content/drive/MyDrive/learning_guide_outputs")
    else:
        root = Path("/content/learning_guide_outputs") if in_colab() else Path("learning_guide_outputs")
    output_dir = root / name
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"出力先ディレクトリ: {output_dir}")
    return output_dir


def download_file(url: str, output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        print(f"既に存在するため再利用します: {output_path}")
        return output_path

    print(f"ダウンロード中: {url}")
    urllib.request.urlretrieve(url, output_path)
    return output_path


def maybe_mount_drive(do_mount: bool = False) -> None:
    if not do_mount or not in_colab():
        return

    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")


def maybe_upload_files() -> list[Path]:
    if not in_colab():
        return []

    from google.colab import files  # type: ignore
    uploaded = files.upload()
    return [Path(name) for name in uploaded.keys()]


def show_image_grid(
    images: list[np.ndarray],
    titles: list[str] | None = None,
    cols: int = 4,
    figsize=(14, 8),
    output_path: Path | None = None,
) -> None:
    if not images:
        return

    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for index, image in enumerate(images, start=1):
        plt.subplot(rows, cols, index)
        plt.imshow(image)
        plt.axis("off")
        if titles and index - 1 < len(titles):
            plt.title(titles[index - 1], fontsize=10)
    plt.tight_layout()
    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_path, dpi=200, bbox_inches="tight")
        print(f"画像グリッドを保存しました: {output_path}")
    plt.show()


def plot_training_curves(
    train_losses: list[float],
    val_losses: list[float],
    train_accs: list[float],
    val_accs: list[float],
    output_path: Path | None = None,
) -> None:
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label="train loss")
    plt.plot(val_losses, label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Loss Curve")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label="train acc")
    plt.plot(val_accs, label="val acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title("Accuracy Curve")
    plt.legend()

    plt.tight_layout()
    if output_path is not None:
        plt.savefig(output_path, dpi=200, bbox_inches="tight")
        print(f"学習曲線を保存しました: {output_path}")
    plt.show()


def plot_grouped_bar(
    labels: list[str],
    series: dict[str, list[float]],
    title: str,
    ylabel: str,
    output_path: Path | None = None,
) -> None:
    if not labels or not series:
        return

    x = np.arange(len(labels))
    width = 0.8 / max(len(series), 1)

    plt.figure(figsize=(max(8, 1.8 * len(labels)), 5))
    for index, (name, values) in enumerate(series.items()):
        offset = (index - (len(series) - 1) / 2) * width
        plt.bar(x + offset, values, width=width, label=name)

    plt.xticks(x, labels, rotation=15)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    if output_path is not None:
        plt.savefig(output_path, dpi=200, bbox_inches="tight")
        print(f"比較グラフを保存しました: {output_path}")
    plt.show()


def denormalize_image(image_tensor: torch.Tensor, mean: tuple[float, ...], std: tuple[float, ...]) -> np.ndarray:
    image = image_tensor.detach().cpu().clone()
    for channel, m, s in zip(image, mean, std):
        channel.mul_(s).add_(m)
    image = image.clamp(0, 1)
    return image.permute(1, 2, 0).numpy()


def save_text(text: str, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(text, encoding="utf-8")
    print(f"テキストを保存しました: {output_path}")


def save_json(data: dict, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"JSON を保存しました: {output_path}")


def count_parameters(model: torch.nn.Module, trainable_only: bool = False) -> int:
    if trainable_only:
        return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return sum(parameter.numel() for parameter in model.parameters())


def format_seconds(seconds: float) -> str:
    minutes, sec = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours >= 1:
        return f"{int(hours)}h {int(minutes)}m {sec:.1f}s"
    if minutes >= 1:
        return f"{int(minutes)}m {sec:.1f}s"
    return f"{seconds:.2f}s"


def sample_video_frames(video_path: Path, num_frames: int, stride: int = 1) -> list[np.ndarray]:
    import cv2

    cap = cv2.VideoCapture(str(video_path))
    frames: list[np.ndarray] = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise RuntimeError(f"動画からフレームを読めませんでした: {video_path}")

    candidate_frames = frames[:: max(1, stride)]
    if len(candidate_frames) == 1:
        return [candidate_frames[0].copy() for _ in range(num_frames)]

    indices = np.linspace(0, len(candidate_frames) - 1, num=num_frames, dtype=int)
    sampled = [candidate_frames[index].copy() for index in indices]
    while len(sampled) < num_frames:
        sampled.append(sampled[-1].copy())
    return sampled


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

ensure_packages({"cv2": "opencv-python-headless"})

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display


# 目的
# - Lucas-Kanade
# - Farneback
# - 観点: 疎 / 密

DEFAULT_VIDEO_URL = (
    "https://huggingface.co/datasets/dyland222/international_amateur_baseball_catcher_video_dataset/"
    "resolve/main/CATCHER_VID_000003.mp4?download=true"
)
MAX_CORNERS = 150
QUALITY_LEVEL = 0.01
MIN_DISTANCE = 7
BLOCK_SIZE = 7
LK_WIN_SIZE = (15, 15)
LK_MAX_LEVEL = 2
LK_MAX_ITER = 10
LK_EPS = 0.03
FARNEBACK_PYR_SCALE = 0.5
FARNEBACK_LEVELS = 3
FARNEBACK_WINSIZE = 15
FARNEBACK_ITERATIONS = 3
FARNEBACK_POLY_N = 5
FARNEBACK_POLY_SIGMA = 1.2
PREVIEW_FRAME_INDICES = {0, 10, 20, 30}
MIN_TRACKED_POINTS_TO_CONTINUE = 10
VIDEO_DISPLAY_WIDTH = 720
FLOW_VARIANTS = [
    {"name": "tight_window", "lk_win_size": (9, 9), "farneback_winsize": 9},
    {"name": "wide_window", "lk_win_size": (21, 21), "farneback_winsize": 21},
]


def run_flow_variant(video_path: Path, output_video: Path, lk_win_size: tuple[int, int], farneback_winsize: int):
    cap = cv2.VideoCapture(str(video_path))
    ok, first_frame = cap.read()
    if not ok:
        raise RuntimeError("最初のフレームが読めませんでした。")

    height, width = first_frame.shape[:2]
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    writer = cv2.VideoWriter(
        str(output_video),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    old_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
    points = cv2.goodFeaturesToTrack(
        old_gray,
        maxCorners=MAX_CORNERS,
        qualityLevel=QUALITY_LEVEL,
        minDistance=MIN_DISTANCE,
        blockSize=BLOCK_SIZE,
    )
    if points is None:
        raise RuntimeError("追跡用の特徴点が見つかりませんでした。")
    initial_point_count = int(len(points))

    trail = np.zeros_like(first_frame)
    colors = np.random.default_rng(0).integers(0, 255, size=(150, 3), dtype=np.uint8)
    lk_params = dict(
        winSize=lk_win_size,
        maxLevel=LK_MAX_LEVEL,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, LK_MAX_ITER, LK_EPS),
    )

    preview_frames = []
    dense_flow_previews = []
    dense_flow_magnitudes = []
    frame_index = 0
    total_lk_seconds = 0.0
    total_dense_flow_seconds = 0.0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        lk_start = time.perf_counter()
        next_points, status, _ = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, points, None, **lk_params)
        total_lk_seconds += time.perf_counter() - lk_start
        if next_points is None or status is None:
            break

        dense_start = time.perf_counter()
        dense_flow = cv2.calcOpticalFlowFarneback(
            old_gray,
            frame_gray,
            None,
            pyr_scale=FARNEBACK_PYR_SCALE,
            levels=FARNEBACK_LEVELS,
            winsize=farneback_winsize,
            iterations=FARNEBACK_ITERATIONS,
            poly_n=FARNEBACK_POLY_N,
            poly_sigma=FARNEBACK_POLY_SIGMA,
            flags=0,
        )
        total_dense_flow_seconds += time.perf_counter() - dense_start
        magnitude, angle = cv2.cartToPolar(dense_flow[..., 0], dense_flow[..., 1])
        dense_flow_magnitudes.append(float(magnitude.mean()))

        good_new = next_points[status.flatten() == 1]
        good_old = points[status.flatten() == 1]
        for idx, (new_point, old_point) in enumerate(zip(good_new, good_old)):
            a, b = new_point.ravel()
            c, d = old_point.ravel()
            color = tuple(int(x) for x in colors[idx % len(colors)])
            cv2.line(trail, (int(a), int(b)), (int(c), int(d)), color, 2)
            cv2.circle(frame, (int(a), int(b)), 4, color, -1)

        output_frame = cv2.add(frame, trail)
        cv2.putText(
            output_frame,
            f"frame={frame_index}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )
        writer.write(output_frame)

        if frame_index in PREVIEW_FRAME_INDICES:
            preview_frames.append(cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB))
            hsv = np.zeros((height, width, 3), dtype=np.uint8)
            hsv[..., 0] = (angle * 90 / np.pi).astype(np.uint8)
            hsv[..., 1] = 255
            hsv[..., 2] = np.clip(magnitude * 20, 0, 255).astype(np.uint8)
            dense_flow_previews.append(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB))

        old_gray = frame_gray
        points = good_new.reshape(-1, 1, 2)
        frame_index += 1
        if len(good_new) < MIN_TRACKED_POINTS_TO_CONTINUE:
            break

    cap.release()
    writer.release()
    return {
        "output_video": output_video,
        "fps": fps,
        "width": width,
        "height": height,
        "processed_frames": frame_index,
        "initial_points": initial_point_count,
        "preview_frames": preview_frames,
        "dense_flow_previews": dense_flow_previews,
        "dense_flow_magnitudes": dense_flow_magnitudes,
        "lk_seconds": total_lk_seconds,
        "farneback_seconds": total_dense_flow_seconds,
    }


def main() -> None:
    overall_start = time.perf_counter()
    maybe_mount_drive(do_mount=True)
    output_dir = get_output_dir("04_optical_flow_tracking", prefer_drive=True)
    print_runtime_info()

    video_path = download_file(DEFAULT_VIDEO_URL, output_dir / "sample_baseball_catcher_view.mp4")
    variant_results = []
    for variant in FLOW_VARIANTS:
        output_video = output_dir / f"optical_flow_tracking_{variant['name']}.mp4"
        result = run_flow_variant(
            video_path=video_path,
            output_video=output_video,
            lk_win_size=variant["lk_win_size"],
            farneback_winsize=variant["farneback_winsize"],
        )
        result["name"] = variant["name"]
        result["lk_win_size"] = variant["lk_win_size"]
        result["farneback_winsize"] = variant["farneback_winsize"]
        variant_results.append(result)

    plt.figure(figsize=(14, 8))
    for index, result in enumerate(variant_results, start=1):
        if result["preview_frames"]:
            plt.subplot(2, 2, 2 * index - 1)
            plt.imshow(result["preview_frames"][0])
            plt.axis("off")
            plt.title(f"L-K ({result['name']})")
        if result["dense_flow_previews"]:
            plt.subplot(2, 2, 2 * index)
            plt.imshow(result["dense_flow_previews"][0])
            plt.axis("off")
            plt.title(f"Farneback ({result['name']})")
    plt.tight_layout()
    plt.savefig(output_dir / "optical_flow_variant_previews.png", dpi=200, bbox_inches="tight")
    plt.show()

    plot_grouped_bar(
        labels=[result["name"] for result in variant_results],
        series={
            "lk seconds": [result["lk_seconds"] for result in variant_results],
            "farneback seconds": [result["farneback_seconds"] for result in variant_results],
        },
        title="Optical Flow: 窓サイズを変えたときの runtime 比較",
        ylabel="seconds",
        output_path=output_dir / "optical_flow_runtime_comparison.png",
    )
    plot_grouped_bar(
        labels=[result["name"] for result in variant_results],
        series={
            "final mean flow magnitude": [
                result["dense_flow_magnitudes"][-1] if result["dense_flow_magnitudes"] else 0.0
                for result in variant_results
            ],
            "initial tracked points": [result["initial_points"] for result in variant_results],
        },
        title="Optical Flow: 窓サイズを変えたときの特徴量比較",
        ylabel="value",
        output_path=output_dir / "optical_flow_feature_comparison.png",
    )

    for result in variant_results:
        print(f"追跡動画を保存しました: {result['output_video']}")
        display(Video(str(result["output_video"]), embed=True, width=VIDEO_DISPLAY_WIDTH))

    overall_seconds = time.perf_counter() - overall_start
    summary_lines = [
        "Optical Flow 比較メモ",
        f"FLOW_VARIANTS: {FLOW_VARIANTS}",
        "Lucas-Kanade は疎な特徴点を追うので、軌跡が分かりやすいです。",
        "Farneback は画面全体の密な動きを出せるので、動きの強い領域を見つけやすいです。",
        f"Notebook total runtime: {format_seconds(overall_seconds)}",
    ]
    for result in variant_results:
        summary_lines.extend(
            [
                f"{result['name']} mean flow measurements: {len(result['dense_flow_magnitudes'])}",
                f"{result['name']} Lucas-Kanade total time: {format_seconds(result['lk_seconds'])}",
                f"{result['name']} Farneback total time: {format_seconds(result['farneback_seconds'])}",
            ]
        )
    save_text("\n".join(summary_lines), output_dir / "summary.txt")
    save_json(
        {
            "config": {
                "max_corners": MAX_CORNERS,
                "quality_level": QUALITY_LEVEL,
                "min_distance": MIN_DISTANCE,
                "block_size": BLOCK_SIZE,
                "lk_win_size": list(LK_WIN_SIZE),
                "lk_max_level": LK_MAX_LEVEL,
                "lk_max_iter": LK_MAX_ITER,
                "lk_eps": LK_EPS,
                "farneback_pyr_scale": FARNEBACK_PYR_SCALE,
                "farneback_levels": FARNEBACK_LEVELS,
                "farneback_winsize": FARNEBACK_WINSIZE,
                "farneback_iterations": FARNEBACK_ITERATIONS,
                "farneback_poly_n": FARNEBACK_POLY_N,
                "farneback_poly_sigma": FARNEBACK_POLY_SIGMA,
                "preview_frame_indices": sorted(PREVIEW_FRAME_INDICES),
                "min_tracked_points_to_continue": MIN_TRACKED_POINTS_TO_CONTINUE,
                "video_display_width": VIDEO_DISPLAY_WIDTH,
                "flow_variants": FLOW_VARIANTS,
            },
            "variant_results": [
                {
                    "name": result["name"],
                    "lk_win_size": list(result["lk_win_size"]),
                    "farneback_winsize": result["farneback_winsize"],
                    "fps": result["fps"],
                    "width": result["width"],
                    "height": result["height"],
                    "processed_frames": result["processed_frames"],
                    "initial_points": result["initial_points"],
                    "num_dense_flow_measurements": len(result["dense_flow_magnitudes"]),
                    "final_dense_flow_magnitude_mean": result["dense_flow_magnitudes"][-1] if result["dense_flow_magnitudes"] else None,
                    "lk_seconds": result["lk_seconds"],
                    "farneback_seconds": result["farneback_seconds"],
                }
                for result in variant_results
            ],
            "runtime": {
                "overall_seconds": overall_seconds,
            },
        },
        output_dir / "metrics.json",
    )


main()


In [ ]:
from pathlib import Path
from IPython.display import Image, Video, display, Markdown

out_dir = None
for candidate in [Path('/content/drive/MyDrive/learning_guide_outputs/04_optical_flow_tracking'), Path('/content/learning_guide_outputs/04_optical_flow_tracking')]:
    if candidate.exists():
        out_dir = candidate
        break

if out_dir is None:
    print('出力ディレクトリが見つかりませんでした。先に上の実験セルを実行してください。')
else:
    print(f'output_dir = {out_dir}')
    summary_path = out_dir / 'summary.txt'
    metrics_path = out_dir / 'metrics.json'
    if summary_path.exists():
        print('\n===== summary.txt =====\n')
        print(summary_path.read_text(encoding='utf-8'))
    if metrics_path.exists():
        print('\n===== metrics.json (先頭 1500 文字) =====\n')
        text = metrics_path.read_text(encoding='utf-8')
        print(text[:1500] + ('...' if len(text) > 1500 else ''))
    for png in sorted(out_dir.glob('*.png')):
        display(Markdown(f'### {png.name}'))
        display(Image(filename=str(png)))
    for mp4 in sorted(out_dir.glob('*.mp4')):
        display(Markdown(f'### {mp4.name}'))
        display(Video(str(mp4), embed=True, width=720))
